# CodeT5+ Fine-tuning for C++ Code Comment Generation

This notebook fine-tunes `Salesforce/codet5p-220m` to add comments to C++ code.

**Task:**
- Input: C++ code WITHOUT comments
- Output: C++ code WITH comments added

**Features:**
- Automatic single / multi-GPU support
- Mixed-precision (FP16) training
- Gradient checkpointing to reduce VRAM
- Resume from last checkpoint if interrupted
- Tracks best checkpoint by validation BLEU
- Full validation loop with BLEU score
- Inference example at the end


## Cell 1 – Install Dependencies


In [ ]:
# ── Install / upgrade core packages ────────────────────────────────────────
# Run only once; Kaggle already ships most of them.
import subprocess, sys

packages = [
    "transformers==4.39.3",
    "datasets>=2.18.0",
    "evaluate>=0.4.1",
    "sacrebleu>=2.3.1",
    "accelerate>=0.29.0",
    "sentencepiece",
    "protobuf",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + packages
)
print("✅ All packages installed / verified.")


## Cell 2 – Imports


In [ ]:
# ── Standard library ───────────────────────────────────────────────────────
import os
import json
import random
import logging
import time
from pathlib import Path
from datetime import datetime

# ── Numeric / ML ───────────────────────────────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast

# ── Hugging Face ──────────────────────────────────────────────────────────
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    get_linear_schedule_with_warmup,
    DataCollatorForSeq2Seq,
)
from datasets import Dataset, DatasetDict
import evaluate

print("✅ Imports complete.")
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    print(f"GPU count       : {n}")
    for i in range(n):
        mem = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({mem:.1f} GB)")


## Cell 3 – Configuration

Edit the variables in this cell to match your environment.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  CONFIGURATION  –  Edit these paths / hyperparameters as needed
# ═══════════════════════════════════════════════════════════════════════════

# ── Dataset paths ─────────────────────────────────────────────────────────
# Data format: {"input": code_without_comments, "output": code_with_comments, "explanation": "..."}
# Task: Given code WITHOUT comments → Generate code WITH comments
DATASET_DIR   = "/kaggle/input/datasets/saffigaming/comment-exlpanation/train_final02.json"   # <-- Change me
# ── Task: Code Comment Generation ──────────────────────────────────────────
# Input: Code WITHOUT comments
# Output: Code WITH comments (learned from train_final02.json)TRAIN_JSON    = os.path.join(DATASET_DIR, "train_final02.json")
VAL_JSON      = os.path.join(DATASET_DIR, "val_final02.json")
TEST_JSON     = os.path.join(DATASET_DIR, "test_final02.json")

# ── Model ─────────────────────────────────────────────────────────────────
# Options: "Salesforce/codet5-base"  |  "Salesforce/codet5-small"
#          "Salesforce/codet5p-220m" (CodeT5+, recommended)
MODEL_NAME    = "Salesforce/codet5p-220m"

# ── Training hyperparameters ──────────────────────────────────────────────
# You can also override these from the command line via environment variables
# e.g.:  BATCH_SIZE=4 EPOCHS=10 python notebook.py
BATCH_SIZE    = int(os.environ.get("BATCH_SIZE",  32))
EPOCHS        = int(os.environ.get("EPOCHS",      8))
LEARNING_RATE = float(os.environ.get("LR",     5e-5))

TRAIN_BATCH_SIZE = 24        # High for fast training
VAL_BATCH_SIZE   = 4         # Low for memory-safe generation (Beam Search)
BATCH_SIZE    = TRAIN_BATCH_SIZE # Default for cfg dict
EPOCHS        = 8
LEARNING_RATE = 5e-5

WARMUP_RATIO  = 0.1          # fraction of total steps used for warmup
GRAD_ACCUM    = 1            # effective batch = BATCH_SIZE × GRAD_ACCUM
MAX_SOURCE_LEN= 512          # max tokens for code (input)
MAX_TARGET_LEN= 512          # max tokens for code with comments + explanation (output)
# Note: For very long functions, increase this or use longer at inference time
WEIGHT_DECAY  = 0.01
SEED          = 42
NUM_BEAMS     = 4            # beam search width during validation / inference

# ── Checkpointing ─────────────────────────────────────────────────────────
OUTPUT_DIR    = "/Volumes/Data/fyp/codet5_cpp_comments"        # <-- Change me
CKPT_LAST     = os.path.join(OUTPUT_DIR, "checkpoint_last")
CKPT_BEST     = os.path.join(OUTPUT_DIR, "checkpoint_best")

# ── Mixed precision ───────────────────────────────────────────────────────
USE_FP16      = torch.cuda.is_available()  # auto-disable on CPU

# ─────────────────────────────────────────────────────────────────────────
# Create output directories
Path(CKPT_LAST).mkdir(parents=True, exist_ok=True)
Path(CKPT_BEST).mkdir(parents=True, exist_ok=True)

print("Configuration:")
cfg = dict(
    MODEL_NAME=MODEL_NAME, BATCH_SIZE=BATCH_SIZE, EPOCHS=EPOCHS,
    LEARNING_RATE=LEARNING_RATE, GRAD_ACCUM=GRAD_ACCUM,
    MAX_SOURCE_LEN=MAX_SOURCE_LEN, MAX_TARGET_LEN=MAX_TARGET_LEN,
    USE_FP16=USE_FP16, OUTPUT_DIR=OUTPUT_DIR,
)
for k, v in cfg.items():
    print(f"  {k:<20}: {v}")


## Cell 4 – Reproducibility


In [ ]:
def seed_everything(seed: int = 42):
    """Set seeds for Python, NumPy, and PyTorch for reproducible results."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Makes CuDNN deterministic (slightly slower but reproducible)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print(f"✅ Seed set to {SEED}")


## Cell 5 – Dataset Validation & Loading


In [ ]:
# ── Helpers ────────────────────────────────────────────────────────────────

def load_json(path: str) -> list:
    """Load a JSON file that is either a list or a dict of records."""
    print(f"  Loading {path} …", end=" ")
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    # Support both list-of-dicts and dict-of-dicts
    if isinstance(data, dict):
        data = list(data.values())
    print(f"{len(data):,} records")
    return data


def clean_code(text: str) -> str:
    """Strip markdown fences ``` and normalise whitespace."""
    if not isinstance(text, str):
        return ""
    # Remove leading/trailing inline code markers
    text = text.strip()
    if text.startswith("`") and text.endswith("`"):
        text = text.strip("`")
    # Remove ```cpp / ``` fences
    lines = text.split("\n")
    cleaned = []
    for line in lines:
        if line.strip().startswith("```"):
            continue
        cleaned.append(line)
    text = "\n".join(cleaned)
    # Collapse excessive blank lines
    import re
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def validate_records(records: list, split_name: str) -> list:
    """
    Validate each record:
    - Must have "input" (code without comments) and "output" (code with comments) keys
    - Optional: "explanation" field
    - Both must be non-empty strings
    Returns list of valid records and prints a summary.
    """
    valid   = []
    skipped = 0
    reasons = {"missing_code": 0, "missing_summary": 0,
               "empty_code": 0, "empty_summary": 0}

    for rec in records:
        # New format: input = code without comments, output = code with comments + explanation
        if "input" not in rec:
            reasons["missing_code"] += 1; skipped += 1; continue
        if "output" not in rec:
            reasons["missing_summary"] += 1; skipped += 1; continue

        code    = clean_code(rec["input"])
        
        # Combine code with comments + explanation
        explanation = rec.get("explanation", "").strip()
        summary = rec["output"].strip()
        
        # Add explanation if available (separated by special marker)
        if explanation:
            # Format: CODE_WITH_COMMENTS

===EXPLANATION===
explanation
            target = f"{summary}

===EXPLANATION===
{explanation}"
        else:
            target = summary

        if not code:
            reasons["empty_code"] += 1; skipped += 1; continue
        if not target:
            reasons["empty_summary"] += 1; skipped += 1; continue

        valid.append({"code": code, "summary": target})

    print(f"  [{split_name}] Total: {len(records):,} | "
          f"Valid: {len(valid):,} | Skipped: {skipped} {reasons}")
    return valid


# ── Load & validate ────────────────────────────────────────────────────────
# Load single file and split into train/val/test
print("Loading dataset from single file …")
raw_data = load_json(DATA_FILE)
print(f"Total records: {len(raw_data):,}")

# Auto-split: 80% train, 10% val, 10% test
import random
random.seed(SEED)
random.shuffle(raw_data)
n = len(raw_data)
train_end = int(0.8 * n)
val_end = int(0.9 * n)

raw_train = raw_data[:train_end]
raw_val   = raw_data[train_end:val_end]
raw_test  = raw_data[val_end:]

print(f"  Train: {len(raw_train):,} | Val: {len(raw_val):,} | Test: {len(raw_test):,}")

print("\nValidating …")
train_records = validate_records(raw_train, "train")
val_records   = validate_records(raw_val,   "val")
test_records  = validate_records(raw_test,  "test")

assert len(train_records) > 0, "❌ Training set is empty after validation!"
assert len(val_records)   > 0, "❌ Validation set is empty after validation!"

# ── Build HuggingFace Dataset objects ─────────────────────────────────────
hf_datasets = DatasetDict({
    "train" : Dataset.from_list(train_records),
    "val"   : Dataset.from_list(val_records),
    "test"  : Dataset.from_list(test_records),
})

print(f"\n✅ DatasetDict ready: {hf_datasets}")

# Quick sanity check on one example
print("\n── Sample from training set ──")
ex = hf_datasets["train"][0]
print(f"CODE    : {ex['code'][:120]} ...")
print(f"SUMMARY : {ex['summary']}")


## Cell 6 – Tokenizer & Tokenization


In [ ]:
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def preprocess(batch):
    """
    Tokenize a batch of (code, summary) pairs.
    Input  → code  (truncated to MAX_SOURCE_LEN)
    Target → summary (truncated to MAX_TARGET_LEN)
    Labels have padding tokens replaced with -100 so they are ignored in loss.
    """
    # Encode source (code)
    model_inputs = tokenizer(
        batch["code"],
        max_length=MAX_SOURCE_LEN,
        padding=False,          # will be handled by the collator
        truncation=True,
    )

    # Encode target (summary) – use the tokenizer's target context manager
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["summary"],
            max_length=MAX_TARGET_LEN,
            padding=False,
            truncation=True,
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


print("Tokenizing datasets …")
tokenized = hf_datasets.map(
    preprocess,
    batched=True,
    batch_size=256,
    remove_columns=["code", "summary"],   # drop raw text columns
    desc="Tokenizing",
)
tokenized.set_format(type="torch")
print(f"✅ Tokenised: {tokenized}")


## Cell 7 – Data Collator & DataLoaders


In [ ]:
# DataCollatorForSeq2Seq pads each batch dynamically (better GPU utilisation)
# and replaces padding ids in labels with -100
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

train_loader = DataLoader(
    tokenized["train"],
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
    num_workers=8,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    tokenized["val"],
    batch_size=VAL_BATCH_SIZE, # Use smaller batch for validation
    shuffle=False,
    collate_fn=data_collator,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    tokenized["test"],
    batch_size=VAL_BATCH_SIZE, # Use smaller batch for testing
    shuffle=False,
    collate_fn=data_collator,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)


print(f"Train batches : {len(train_loader):,}")
print(f"Val   batches : {len(val_loader):,}")
print(f"Test  batches : {len(test_loader):,}")
print(f"Effective batch size : {BATCH_SIZE * GRAD_ACCUM}")


## Cell 8 – Model Setup


In [ ]:
# ── Device setup ───────────────────────────────────────────────────────────
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpus    = torch.cuda.device_count()
print(f"Device : {device}  |  GPUs : {n_gpus}")

# ── Load model ─────────────────────────────────────────────────────────────
print(f"Loading model: {MODEL_NAME} …")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# ── Gradient checkpointing ─ saves ~30-40% VRAM, slight speed cost ─────────
model.gradient_checkpointing_enable()

# ── Multi-GPU wrapping ─ transparent to the training loop ──────────────────
if n_gpus > 1:
    print(f"Wrapping model in DataParallel across {n_gpus} GPUs.")
    model = nn.DataParallel(model)

model = model.to(device)

total_params  = sum(p.numel() for p in model.parameters())
train_params  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {train_params:,}")


## Cell 9 – Optimizer, Scheduler & Scaler


In [ ]:
# AdamW with weight-decay applied to everything except biases and LayerNorm
no_decay     = ["bias", "LayerNorm.weight"]
param_groups = [
    {
        "params": [p for n, p in model.named_parameters()
                   if not any(nd in n for nd in no_decay)],
        "weight_decay": WEIGHT_DECAY,
    },
    {
        "params": [p for n, p in model.named_parameters()
                   if any(nd in n for nd in no_decay)],
        "weight_decay": 0.0,
    },
]

optimizer = torch.optim.AdamW(param_groups, lr=LEARNING_RATE)

total_steps  = (len(train_loader) // GRAD_ACCUM) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

# Mixed-precision scaler – skips update if NaN/Inf gradients are detected
scaler = GradScaler(enabled=USE_FP16)

print(f"Total optimisation steps  : {total_steps:,}")
print(f"Warmup steps              : {warmup_steps:,}")
print(f"FP16 enabled              : {USE_FP16}")


## Cell 10 – Checkpoint Utilities


In [ ]:
def save_checkpoint(path: str, epoch: int, model, optimizer,
                    scheduler, scaler, best_bleu: float,
                    history: dict):
    """
    Save model weights + training state so training can resume exactly.
    The 'model' might be wrapped in DataParallel; unwrap before saving.
    """
    Path(path).mkdir(parents=True, exist_ok=True)
    raw_model = model.module if hasattr(model, "module") else model

    # Save Hugging Face model and tokenizer (for easy re-loading)
    raw_model.save_pretrained(path)
    tokenizer.save_pretrained(path)

    # Save optimizer / scheduler / scaler state for exact resume
    torch.save({
        "epoch"     : epoch,
        "optimizer" : optimizer.state_dict(),
        "scheduler" : scheduler.state_dict(),
        "scaler"    : scaler.state_dict(),
        "best_bleu" : best_bleu,
        "history"   : history,
    }, os.path.join(path, "training_state.pt"))
    print(f"  💾 Checkpoint saved → {path}")


def load_checkpoint(path: str, model, optimizer, scheduler, scaler):
    """
    Load a checkpoint.  Returns (start_epoch, best_bleu, history).
    Returns (0, 0.0, {}) if no checkpoint is found.
    """
    state_file = os.path.join(path, "training_state.pt")
    if not os.path.isfile(state_file):
        print(f"  No checkpoint found at {path}. Starting from scratch.")
        return 0, 0.0, {"train_loss": [], "val_loss": [], "val_bleu": []}

    print(f"  📂 Loading checkpoint from {path} …")
    state = torch.load(state_file, map_location="cpu")

    # Load model weights (handle DataParallel wrapping)
    hf_model = AutoModelForSeq2SeqLM.from_pretrained(path)
    raw_model = model.module if hasattr(model, "module") else model
    raw_model.load_state_dict(hf_model.state_dict())
    del hf_model

    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    scaler.load_state_dict(state["scaler"])

    start_epoch = state["epoch"] + 1      # resume AFTER the saved epoch
    best_bleu   = state["best_bleu"]
    history     = state["history"]
    print(f"  ✅ Resuming from epoch {start_epoch}  |  Best BLEU so far: {best_bleu:.4f}")
    return start_epoch, best_bleu, history


print("✅ Checkpoint utilities defined.")


## Cell 11 – Validation Helper (Loss + BLEU)


In [ ]:
# Load the sacrebleu metric  (corpus-level BLEU)
bleu_metric = evaluate.load("sacrebleu")


def run_validation(model, loader, tokenizer, device,
                   max_gen_len=MAX_TARGET_LEN,
                   num_beams=NUM_BEAMS,
                   max_val_batches=None):
    """
    Evaluate the model on `loader`.
    Returns:
        val_loss (float)  – mean cross-entropy loss
        bleu     (float)  – corpus BLEU score

    Set max_val_batches to a small number (e.g. 50) for a fast sanity check.
    """
    model.eval()
    raw_model = model.module if hasattr(model, "module") else model

    total_loss = 0.0
    n_batches  = 0
    predictions, references = [], []

    with torch.no_grad():
        for i, batch in enumerate(loader):
            if max_val_batches and i >= max_val_batches:
                break

            batch = {k: v.to(device) for k, v in batch.items()}

            # ── Loss ──────────────────────────────────────────────────────
            with autocast(enabled=USE_FP16):
                outputs = model(**batch)
            # DataParallel returns mean-reduced loss automatically
            total_loss += outputs.loss.mean().item()
            n_batches  += 1

            # ── Generation for BLEU ───────────────────────────────────────
            generated_ids = raw_model.generate(
                input_ids      = batch["input_ids"],
                attention_mask = batch["attention_mask"],
                max_length     = max_gen_len,
                num_beams      = num_beams,
                early_stopping = True,
            )

            # Decode predictions
            preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

            # Decode references (labels; -100 → pad_token_id for decoding)
            label_ids = batch["labels"].clone()
            label_ids[label_ids == -100] = tokenizer.pad_token_id
            refs = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

            predictions.extend(preds)
            references.extend([[r] for r in refs])   # sacrebleu expects list-of-lists

    val_loss = total_loss / max(n_batches, 1)
    bleu_result = bleu_metric.compute(
        predictions=predictions,
        references=references,
    )
    bleu_score = bleu_result["score"]   # 0–100 scale

    return val_loss, bleu_score


print("✅ Validation helper defined.")


## Cell 12 – Resume Check


In [ ]:
# ── Try to resume from the last checkpoint ─────────────────────────────────
# If checkpoint_last exists and has a training_state.pt, resume from there.
# Otherwise start fresh.

print("Checking for existing checkpoint …")
start_epoch, best_bleu, history = load_checkpoint(
    CKPT_LAST, model, optimizer, scheduler, scaler
)

if not history:
    history = {"train_loss": [], "val_loss": [], "val_bleu": []}


## Cell 13 – Training Loop


In [ ]:
# ── Training loop ──────────────────────────────────────────────────────────
#
# Memory safety: if an OOM error occurs mid-epoch the loop catches it,
# reduces the effective batch (via gradient accumulation) and continues.
# The model is saved after every epoch so progress is never lost.

os.environ["TOKENIZERS_PARALLELISM"] = "false"

LOG_EVERY_N_STEPS = 20    # print loss every N optimiser steps

print("\n" + "═" * 70)
print(f" Training {MODEL_NAME.split('/')[-1]} for {EPOCHS} epochs")
print(f" Starting epoch: {start_epoch + 1} / {EPOCHS}")
print("═" * 70 + "\n")

for epoch in range(start_epoch, EPOCHS):
    epoch_start = time.time()
    model.train()
    optimizer.zero_grad()

    running_loss = 0.0
    step_loss    = 0.0
    opt_step     = 0       # number of optimiser steps this epoch

    print(f"\n── Epoch {epoch + 1} / {EPOCHS} ─────────────────────────────────")

    for batch_idx, batch in enumerate(train_loader):
        try:
            batch = {k: v.to(device) for k, v in batch.items()}

            # ── Forward pass with optional FP16 ─────────────────────────
            with autocast(enabled=USE_FP16):
                outputs = model(**batch)
                # DataParallel returns per-GPU losses → take mean
                loss    = outputs.loss.mean() / GRAD_ACCUM

            # ── Backward pass ────────────────────────────────────────────
            scaler.scale(loss).backward()

            step_loss    += loss.item() * GRAD_ACCUM   # un-scale for logging
            running_loss += loss.item() * GRAD_ACCUM

            # ── Optimiser step every GRAD_ACCUM mini-batches ─────────────
            if (batch_idx + 1) % GRAD_ACCUM == 0:
                # Gradient clipping to prevent exploding gradients
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
                opt_step += 1

                if opt_step % LOG_EVERY_N_STEPS == 0:
                    avg = step_loss / LOG_EVERY_N_STEPS
                    lr  = scheduler.get_last_lr()[0]
                    print(f"  Epoch {epoch+1} | step {opt_step:>5} "
                          f"| loss {avg:.4f} | lr {lr:.2e}")
                    step_loss = 0.0

        except torch.cuda.OutOfMemoryError:
            # ── OOM recovery ─────────────────────────────────────────────
            print(f"  ⚠️  OOM at batch {batch_idx}! "
                  "Clearing cache and skipping this batch.")
            torch.cuda.empty_cache()
            optimizer.zero_grad()
            continue

    # ── End-of-epoch metrics calculation ──────────────────────────────────
    n_train_batches = len(train_loader)
    epoch_loss = running_loss / max(n_train_batches, 1)
    # ── ALWAYS save the LAST checkpoint BEFORE validation ────────────────
    # This ensures progress is saved even if validation crashes.
    save_checkpoint(
        CKPT_LAST, epoch, model, optimizer, scheduler, scaler,
        best_bleu, history
    )
    # ── Memory-safe Validation ───────────────────────────────────────────
    print(f"\n  Clearing cache and starting validation …")
    torch.cuda.empty_cache() # Clear training memory before starting generation
    
    val_loss, val_bleu = run_validation(
        model, val_loader, tokenizer, device,
        # max_val_batches=100, # Optional: uncomment for faster validation
    )
    elapsed = time.time() - epoch_start
    print(f"\n  ┌─ Epoch {epoch+1} summary ────────────────")
    print(f"  │  Train loss : {epoch_loss:.4f}")
    print(f"  │  Val   loss : {val_loss:.4f}")
    print(f"  │  Val  BLEU  : {val_bleu:.2f}")
    print(f"  │  Time       : {elapsed:.0f}s")
    print(f"  └─────────────────────────────────────────")
    # Record history
    history["train_loss"].append(epoch_loss)
    history["val_loss"].append(val_loss)
    history["val_bleu"].append(val_bleu)
    # ── Save BEST checkpoint whenever BLEU improves ──────────────────────
    if val_bleu > best_bleu:
        improvement = val_bleu - best_bleu
        best_bleu   = val_bleu
        save_checkpoint(
            CKPT_BEST, epoch, model, optimizer, scheduler, scaler,
            best_bleu, history
        )
        print(f"  🏆 New best BLEU: {best_bleu:.2f}  (+{improvement:.2f})")

print("\n" + "═" * 70)
print(f" Training complete!  Best validation BLEU: {best_bleu:.2f}")
print("═" * 70)


## Cell 14 – Training History / Plots


In [ ]:
import matplotlib.pyplot as plt

epochs_done = list(range(1, len(history["train_loss"]) + 1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Loss curves
ax1.plot(epochs_done, history["train_loss"], label="Train Loss", marker="o")
ax1.plot(epochs_done, history["val_loss"],   label="Val Loss",   marker="o")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training & Validation Loss")
ax1.legend()
ax1.grid(True)

# BLEU curve
ax2.plot(epochs_done, history["val_bleu"], label="Val BLEU", color="green", marker="o")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("BLEU Score")
ax2.set_title("Validation BLEU Score")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, "training_curves.png")
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"Plot saved → {plot_path}")


## Cell 15 – Test-set Evaluation


In [ ]:
# Load the BEST checkpoint for final evaluation
print("Loading best checkpoint for test evaluation …")
best_model = AutoModelForSeq2SeqLM.from_pretrained(CKPT_BEST)
best_model = best_model.to(device)

test_loss, test_bleu = run_validation(
    best_model, test_loader, tokenizer, device
)

print(f"\n{'═'*40}")
print(f"  Test Loss : {test_loss:.4f}")
print(f"  Test BLEU : {test_bleu:.2f}")
print(f"{'═'*40}")


## Handling Long Code Snippets

If your code is very long:
1. **Increase max_target_len**: Pass a larger value to `generate_code_with_comments(..., max_target_len=1024)`
2. **For very long functions**: Consider splitting into smaller logical units
3. **Memory trade-off**: Longer sequences use more GPU memory


## Cell 16 – Inference Example

Load the best checkpoint and run it on raw C++ code WITHOUT comments.
The model will generate code WITH comments added.


In [ ]:
def generate_code_with_comments(code_snippet: str,
                     model,
                     tokenizer,
                     device,
                     max_source_len: int = MAX_SOURCE_LEN,
                     max_target_len: int = 768,  # Allow longer for inference (override if needed)
                     num_beams: int = NUM_BEAMS):
    """
    Generate C++ code WITH comments AND explanation for the given C++ code snippet (without comments).

    Args:
        code_snippet  : Raw C++ source code WITHOUT comments as a string.
        model         : Loaded Seq2Seq model.
        tokenizer     : Corresponding tokenizer.
        device        : torch.device to run inference on.
        max_source_len: Max input token length.
        max_target_len: Max output token length.
        num_beams     : Beam search width (higher = better but slower).

    Returns:
        tuple: (code_with_comments, explanation)
    """
    model.eval()
    raw_model = model.module if hasattr(model, "module") else model

    inputs = tokenizer(
        code_snippet,
        return_tensors="pt",
        max_length=max_source_len,
        truncation=True,
        padding=False,
    )
    input_ids      = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    with torch.no_grad():
        generated_ids = raw_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=max_target_len,
            num_beams=num_beams,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )

    full_output = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    
    # Parse code and explanation from output
    if "===EXPLANATION===" in full_output:
        code_with_comments, explanation = full_output.split("===EXPLANATION===", 1)
        code_with_comments = code_with_comments.strip()
        explanation = explanation.strip()
    else:
        code_with_comments = full_output
        explanation = ""
    
    return code_with_comments, explanation


# ── Demo ─────────────────────────────────────────────────────────────────
# Example: Code WITHOUT comments → Model adds comments + explanation
cpp_examples = [
    # Example 1: Simple function
    """
#include <iostream>
using namespace std;

int factorial(int n) {
    if (n < 0) throw invalid_argument("negative");
    int result = 1;
    for (int i = 2; i <= n; ++i) result *= i;
    return result;
}
""",
    # Example 2: Binary search
    """
int binarySearch(int arr[], int n, int key) {
    int left = 0, right = n - 1;
    while (left <= right) {
        int mid = left + (right - left) / 2;
        if (arr[mid] == key) return mid;
        else if (arr[mid] < key) left = mid + 1;
        else right = mid - 1;
    }
    return -1;
}
""",
]

for i, cpp in enumerate(cpp_examples, 1):
    code_with_comments, explanation = generate_code_with_comments(cpp.strip(), best_model, tokenizer, device)
    print(f"{'='*60}")
    print(f" EXAMPLE {i}")
    print(f"{'─'*60}")
    print(f"=== INPUT: Code WITHOUT comments ===")
    print(cpp.strip())
    print(f"
=== OUTPUT: Code WITH comments ===")
    print(code_with_comments)
    print(f"
=== EXPLANATION ===")
    print(explanation)
    print()

## Cell 17 – Quick Inference Helper (post-training)

Use this cell anytime to load a saved checkpoint and run inference without re-running training.


In [ ]:
# ── Load either best or last checkpoint ────────────────────────────────────
LOAD_CKPT = CKPT_BEST   # or CKPT_LAST

inf_tokenizer = AutoTokenizer.from_pretrained(LOAD_CKPT)
inf_model     = AutoModelForSeq2SeqLM.from_pretrained(LOAD_CKPT)
inf_model     = inf_model.to(device)
inf_model.eval()

# Paste any C++ code WITHOUT comments below ↓
USER_CODE = """
#include <iostream>
using namespace std;

double mean(const vector<double>& v) {
    double sum = 0;
    for (double x : v) sum += x;
    return sum / v.size();
}
"""

inputs = inf_tokenizer(
    USER_CODE.strip(),
    return_tensors="pt",
    max_length=MAX_SOURCE_LEN,
    truncation=True,
).to(device)

with torch.no_grad():
    out_ids = inf_model.generate(
        **inputs,
        max_length=768,  # Increase if your code is very long
        num_beams=NUM_BEAMS,
        early_stopping=True,
        no_repeat_ngram_size=3,
    )

full_output = inf_tokenizer.decode(out_ids[0], skip_special_tokens=True)

# Parse code and explanation
if "===EXPLANATION===" in full_output:
    output_code, output_explanation = full_output.split("===EXPLANATION===", 1)
    output_code = output_code.strip()
    output_explanation = output_explanation.strip()
else:
    output_code = full_output
    output_explanation = ""

print("=" * 60)
print("=== INPUT: Code WITHOUT comments ===")
print(USER_CODE.strip())
print()
print("=== OUTPUT: Code WITH comments ===")
print(output_code)
print()
print("=== EXPLANATION ===")
print(output_explanation)
print("=" * 60)